<a href="https://colab.research.google.com/github/OlhaZahrebelna/certflow-rag-assistant/blob/main/src/ingestion/chunker.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
pip install pyyaml

In [2]:
!pip install langchain-text-splitters

In [3]:
!git clone https://github.com/OlhaZahrebelna/certflow-rag-assistant.git

Cloning into 'certflow-rag-assistant'...
remote: Enumerating objects: 135, done.
remote: Counting objects: 100% (135/135), done.
remote: Compressing objects: 100% (107/107), done.
remote: Total 135 (delta 37), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (135/135), 227.68 KiB | 4.74 MiB/s, done.
Resolving deltas: 100% (37/37), done.


In [4]:
%cd /content/certflow-rag-assistant

/content/certflow-rag-assistant


In [5]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
import re
import json
from pathlib import Path
import yaml

In [6]:
data_dir = Path("data/raw/source_markdown")

print(data_dir.exists())

True


In [7]:
files = list(data_dir.glob("*.md"))

for file in files:
    print(file.name)

print(f"\nTotal documents: {len(files)}")

09_frequently_asked_questions.md
04_account_fields_and_validation_rules.md
06_address_certification_and_duplicate_prevention.md
02_roles_and_responsibilities.md
07_request_types_and_change_management.md
01_account_certification_overview.md
05_source_hierarchy_and_evidence.md
03_end_to_end_certification_workflow.md
10_policy_change_log.md
08_quality_review_exceptions_and_escalations.md

Total documents: 10


In [8]:
def load_markdown_document(file_path: str) -> tuple[dict, str]:
    """
    Load metadata and content from a Markdown document.

    Expected format:

    ---
    document_id: ACD-KB-001
    title: Account Data Certification Overview
    ...
    ---

    # Account Data Certification Overview
    ...
    """

    path = Path(file_path)

    text = path.read_text(encoding="utf-8")

    # Split YAML front matter from Markdown content
    parts = text.split("---", 2)

    if len(parts) != 3:
        raise ValueError(
            f"File {file_path} does not contain valid YAML front matter."
        )

    metadata_text = parts[1]
    content = parts[2].strip()

    metadata = yaml.safe_load(metadata_text)

    return metadata, content

In [9]:
def split_into_sections(content: str) -> list[dict]:
    """
    Split Markdown document by level-2 headings (##).
    """

    pattern = r"(?m)^##\s+(.+)$"

    matches = list(re.finditer(pattern, content))

    sections = []

    for i, match in enumerate(matches):

        section_title = match.group(1).strip()

        start = match.end()

        if i + 1 < len(matches):
            end = matches[i + 1].start()
        else:
            end = len(content)

        section_content = content[start:end].strip()

        sections.append(
            {
                "section_title": section_title,
                "content": section_content,
            }
        )

    return sections

In [10]:
def create_chunks(metadata: dict, sections: list[dict]) -> list[dict]:
    chunks = []
    chunk_counter = 1

    for section_index, section in enumerate(sections):

        section_parts = text_splitter.split_text(section["content"])

        for part_index, part in enumerate(section_parts):

            chunk_id = (
                f"{metadata['document_id']}-"
                f"chunk-{chunk_counter:03d}"
            )

            chunk = {
                "chunk_id": chunk_id,
                "content": part,
                "metadata": {
                    **metadata,
                    "section": section["section_title"],
                    "section_number": section_index + 1,
                    "section_part": part_index + 1
                },
            }

            chunks.append(chunk)
            chunk_counter += 1

    return chunks

In [11]:
def process_document(file_path: str) -> list[dict]:

    metadata, content = load_markdown_document(file_path)

    sections = split_into_sections(content)

    chunks = create_chunks(
        metadata=metadata,
        sections=sections
    )

    return chunks

In [13]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
    add_start_index=True,
)

all_chunks = []

for file_path in data_dir.glob("*.md"):
    chunks = process_document(file_path)

    all_chunks.extend(chunks)

    print(f"{file_path.name}: {len(chunks)} chunks")

print(f"\nTotal chunks: {len(all_chunks)}")

09_frequently_asked_questions.md: 5 chunks
04_account_fields_and_validation_rules.md: 10 chunks
06_address_certification_and_duplicate_prevention.md: 9 chunks
02_roles_and_responsibilities.md: 8 chunks
07_request_types_and_change_management.md: 7 chunks
01_account_certification_overview.md: 8 chunks
05_source_hierarchy_and_evidence.md: 9 chunks
03_end_to_end_certification_workflow.md: 10 chunks
10_policy_change_log.md: 6 chunks
08_quality_review_exceptions_and_escalations.md: 9 chunks

Total chunks: 81


In [14]:
output_dir = Path("data/processed")
output_dir.mkdir(parents=True, exist_ok=True)

output_path = output_dir / "chunks.json"

with open(output_path, "w", encoding="utf-8") as f:
    json.dump(
        all_chunks,
        f,
        ensure_ascii=False,
        indent=2,
        default=str
    )

print(f"Saved {len(all_chunks)} chunks to {output_path}")

Saved 81 chunks to data/processed/chunks.json


In [15]:
print(output_path.exists())
print(output_path)

True
data/processed/chunks.json
